In [1]:
import pandas as pd

df = pd.read_csv("../incidents.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (50000, 9)

Columns:
['incident_id', 'opened_at', 'priority', 'category', 'assignment_group', 'resolution_minutes', 'reassignment_count', 'sla_target', 'sla_breached']


,incident_id,opened_at,priority,category,assignment_group,resolution_minutes,reassignment_count,sla_target,sla_breached
0,INC00001,2025-01-01 00:00:00.000000,P3,Hardware,Hardware Team,157,3,1440,False
1,INC00002,2025-01-01 00:10:29.004580,P3,Network,Network Team,982,3,1440,False
2,INC00003,2025-01-01 00:20:58.009160,P2,Database,DB Team,740,0,480,True
3,INC00004,2025-01-01 00:31:27.013740,P1,Software,App Team,321,1,240,True
4,INC00005,2025-01-01 00:41:56.018320,P1,Database,DB Team,873,0,240,True


In [2]:
df["opened_at"] = pd.to_datetime(df["opened_at"])

df["hour_of_day"] = df["opened_at"].dt.hour
df["day_of_week"] = df["opened_at"].dt.dayofweek
df["day_of_month"] = df["opened_at"].dt.day
df["month"] = df["opened_at"].dt.month
df["week_of_year"] = df["opened_at"].dt.isocalendar().week.astype(int)

df["is_weekend"] = df["day_of_week"] >= 5
df["is_business_hours"] = df["hour_of_day"].between(9, 17)

df["is_month_start"] = df["opened_at"].dt.is_month_start
df["is_month_end"] = df["opened_at"].dt.is_month_end

df["high_reassignment"] = df["reassignment_count"] >= 2

df["high_priority"] = df["priority"].isin(["P1", "P2"])

df["low_priority"] = df["priority"] == "P4"

print("Feature engineering completed.")

Feature engineering completed.


In [3]:
from sklearn.model_selection import train_test_split

# Features used by the model
feature_columns = [
    "priority",
    "category",
    "assignment_group",
    "reassignment_count",
    "hour_of_day",
    "day_of_week",
    "day_of_month",
    "month",
    "week_of_year",
    "is_weekend",
    "is_business_hours",
    "is_month_start",
    "is_month_end",
    "high_reassignment",
    "high_priority",
    "low_priority"
]

X = df[feature_columns]
y = df["sla_breached"]

print("Number of features:", len(feature_columns))
print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

Number of features: 16
X shape: (50000, 16)
y shape: (50000,)

Training rows: 40000
Testing rows: 10000


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "priority",
    "category",
    "assignment_group"
]

numeric_features = [
    "reassignment_count",
    "hour_of_day",
    "day_of_week",
    "day_of_month",
    "month",
    "week_of_year",
    "is_weekend",
    "is_business_hours",
    "is_month_start",
    "is_month_end",
    "high_reassignment",
    "high_priority",
    "low_priority"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

logistic_model.fit(X_train, y_train)

logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

logistic_pr_auc = average_precision_score(
    y_test,
    logistic_probabilities
)

print("Logistic Regression PR-AUC:", logistic_pr_auc)

Logistic Regression PR-AUC: 0.7517742920357123


In [6]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

random_forest_model.fit(X_train, y_train)

random_forest_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

random_forest_pr_auc = average_precision_score(
    y_test,
    random_forest_probabilities
)

print("Random Forest PR-AUC:", random_forest_pr_auc)

Random Forest PR-AUC: 0.7249250287476064


In [7]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ))
    ]
)

gradient_boosting_model.fit(X_train, y_train)

gradient_boosting_probabilities = (
    gradient_boosting_model.predict_proba(X_test)[:, 1]
)

gradient_boosting_pr_auc = average_precision_score(
    y_test,
    gradient_boosting_probabilities
)

print("Gradient Boosting PR-AUC:", gradient_boosting_pr_auc)

Gradient Boosting PR-AUC: 0.7522895871005142


In [8]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "PR-AUC": [
        logistic_pr_auc,
        random_forest_pr_auc,
        gradient_boosting_pr_auc
    ]
})

comparison = comparison.sort_values(
    by="PR-AUC",
    ascending=False
).reset_index(drop=True)

print(comparison)

                 Model    PR-AUC
0    Gradient Boosting  0.752290
1  Logistic Regression  0.751774
2        Random Forest  0.724925


## Model Comparison and Recommendation

The three models were compared using PR-AUC because the goal is to evaluate how well the models identify SLA breaches.

| Model | PR-AUC |
|---|---:|
| Gradient Boosting | 0.7523 |
| Logistic Regression | 0.7518 |
| Random Forest | 0.7249 |

Gradient Boosting achieved the highest PR-AUC at approximately 0.7523, followed very closely by Logistic Regression at approximately 0.7518. Random Forest performed lower at approximately 0.7249. Based on PR-AUC alone, Gradient Boosting is the recommended model. However, the difference between Gradient Boosting and Logistic Regression is very small, so Logistic Regression may still be attractive because it is simpler and easier to interpret. The final model choice should also consider recall, precision, operational costs, interpretability, and deployment requirements.